In [3]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# Pretrained model

In [4]:
v_model_name = "distilbert-base-uncased-finetuned-sst-2-english"

v_model = AutoModelForSequenceClassification.from_pretrained(v_model_name)
v_tokenizer = AutoTokenizer.from_pretrained(v_model_name)

v_classifier = pipeline("sentiment-analysis", model=v_model_name, tokenizer=v_tokenizer)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# Use and tokenizer

In [15]:
x_train = [
    "Just have trying, don't end up like me",
    "I'll see you around"
]

In [8]:
v_list_result = v_classifier(x_train)

for result in v_list_result:
    print(result)

{'label': 'NEGATIVE', 'score': 0.9695118069648743}
{'label': 'POSITIVE', 'score': 0.9998428821563721}


In [13]:
v_tokens = v_tokenizer.tokenize(x_train[0])
v_token_ids = v_tokenizer.convert_tokens_to_ids(v_tokens)
input_ids = v_tokenizer(DOCUMENTS[0])
print(f"Token: {v_tokens}")
print(f"Token ID: {v_token_ids}")
print(f"Input token: {input_ids}")


Token: ['just', 'have', 'trying', ',', 'don', "'", 't', 'end', 'up', 'like', 'me']
Token ID: [2074, 2031, 2667, 1010, 2123, 1005, 1056, 2203, 2039, 2066, 2033]
Input token: {'input_ids': [101, 2074, 2031, 2667, 1010, 2123, 1005, 1056, 2203, 2039, 2066, 2033, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


# Prediction

In [17]:
# Tokenizer text before get it into model
v_batch = v_tokenizer(
    x_train,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt" # PyTorch
)
print(v_batch)

{'input_ids': tensor([[ 101, 2074, 2031, 2667, 1010, 2123, 1005, 1056, 2203, 2039, 2066, 2033,
          102],
        [ 101, 1045, 1005, 2222, 2156, 2017, 2105,  102,    0,    0,    0,    0,
            0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])}


In [26]:
with torch.no_grad():
    v_output = v_model(**v_batch, labels=torch.tensor([0, 1]))
    print(v_output)
    v_prediction = F.softmax(v_output.logits, dim=1)
    print(v_prediction)
    v_label = v_prediction.argmax(dim=1)
    print(v_label)

    # id2label is maybe only available in AutoModelForSequenceClassification
    v_label = [v_model.config.id2label[label_idx] for label_idx in v_label.tolist()]
    print(v_label)

SequenceClassifierOutput(loss=tensor(0.0156), logits=tensor([[ 1.8357, -1.6238],
        [-4.1782,  4.5802]]), hidden_states=None, attentions=None)
tensor([[9.6951e-01, 3.0488e-02],
        [1.5710e-04, 9.9984e-01]])
tensor([0, 1])
['NEGATIVE', 'POSITIVE']


In [27]:
print(torch.tensor([9.6951e-01, 3.0488e-02]).sum())

tensor(1.0000)
